# 01 MediSim Data Pipeline: Collection & Preparation

This notebook documents the acquisition and processing of the **IU X-Ray dataset** for the **Multimodal Diagnostic Assistant**.

## 1. Environment & Hardware Detection
We ensure the environment is compatible with **M2/M4 Mac**, **Google Colab**, and **AIT Puffer Server** as per the Project Intended requirements.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import platform
import torch
from pathlib import Path

def setup_environment():
    print("--- MediSim Environment Setup ---")

    # 1. Detect Environment
    is_colab = "COLAB_GPU" in os.environ
    print(f"Platform: {platform.system()} {platform.release()}")
    print(f"Google Colab: {'Yes' if is_colab else 'No'}")

    # 2. Mount Google Drive if in Colab
    if is_colab:
        from google.colab import drive
        drive.mount('/content/drive')
        # Set working directory to the notebooks folder in Drive
        # This ensures that '../data/' paths work correctly
        project_path = '/content/drive/My Drive/AIT/NLP/NLP_Project/MediSim/notebooks'
        if os.path.exists(project_path):
            os.chdir(project_path)
            print(f"Changed directory to: {os.getcwd()}")
        else:
            print(f"WARNING: Project path not found: {project_path}")

    # 3. Check Hardware Acceleration
    device = "cpu"
    if torch.backends.mps.is_available():
        device = "mps"
        print("Hardware Acceleration: Apple Metal (MPS) Available")
    elif torch.cuda.is_available():
        device = "cuda"
        print("Hardware Acceleration: NVIDIA CUDA Available")
    else:
        print("Hardware Acceleration: CPU only")

    print(f"Default Device: {device}")
    print("---------------------------------")
    return device

device = setup_environment()

--- MediSim Environment Setup ---
Platform: Darwin 25.4.0
Google Colab: No
Hardware Acceleration: Apple Metal (MPS) Available
Default Device: mps
---------------------------------


## 2. Data Collection (IU X-Ray)
The primary dataset is the **IU X-Ray collection**. We use the Kaggle API for direct acquisition.

In [ ]:
# !pip install kaggle
# Note: Ensure you have kaggle.json in ~/.kaggle/
# !kaggle datasets download -d raddar/chest-xrays-indiana-university -p ../data/ --unzip

## 3. Data Cleaning & Label Extraction
We parse the reports to extract **Findings** and **Impression**, and filter for **Frontal** X-ray projections.

In [2]:
import pandas as pd
import re

# Paths (relative to MediSim/notebooks)
REPORTS_PATH = "../data/indiana_reports.csv"
PROJECTIONS_PATH = "../data/indiana_projections.csv"
IMAGES_DIR = "../data/images"

def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'xxxx', '', text) # Remove de-identified placeholders
    return " ".join(text.split())

# Load and clean
reports_df = pd.read_csv(REPORTS_PATH)
projections_df = pd.read_csv(PROJECTIONS_PATH)

reports_df['findings'] = reports_df['findings'].apply(clean_text)
frontal_projs = projections_df[projections_df['projection'] == 'Frontal'].copy()
merged_df = pd.merge(reports_df, frontal_projs, on='uid')
merged_df = merged_df[merged_df['findings'] != ""].dropna(subset=['filename'])

# Simple Labeling based on MeSH headers
TARGET_CLASSES = ['normal', 'cardiomegaly', 'effusion', 'pneumonia', 'opacity', 'hypoinflation', 'hyperdistention', 'tortuous', 'degenerative', 'granulomatous', 'atherosclerosis', 'calcinosis', 'emphysema', 'nodule', 'fracture']
def extract_label(meshes):
    if not isinstance(meshes, str): return "other"
    meshes = meshes.lower()
    for cls in TARGET_CLASSES:
        if cls in meshes: return cls
    return "other"

merged_df['label'] = merged_df['MeSH'].apply(extract_label)
train_df = merged_df[merged_df['label'] != "other"].copy()

METADATA_SAVE_PATH = "../data/processed_metadata.csv"
train_df.to_csv(METADATA_SAVE_PATH, index=False)
print(f"Dataset prepared with {len(train_df)} frontal scans.")
train_df['label'].value_counts()

Dataset prepared with 2771 frontal scans.


label
normal             1207
cardiomegaly        273
opacity             273
degenerative        234
hyperdistention     159
hypoinflation       157
tortuous            111
calcinosis           78
effusion             76
granulomatous        53
atherosclerosis      40
fracture             34
nodule               30
pneumonia            30
emphysema            16
Name: count, dtype: int64